# Generate the calibration / ensembling dataset

Builds the synthetic dataset used by `calibration_ensembling.ipynb`: per-sample predicted probabilities from two predictors A and B, plus the binary outcome.

In [ ]:
import numpy as np

## Latent features and outcomes

Each sample has two latent features. The true positive-class probability is a logistic function of those features, and the binary label `y` is sampled from a Bernoulli at that probability.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def true_logit(x1, x2):
    return 1.6 * x1 + 1.2 * x2 - 0.3

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0.0, 1.0, size=n)
    x2 = rng.normal(0.0, 1.0, size=n)
    p_true = sigmoid(true_logit(x1, x2))
    y = (rng.uniform(size=n) < p_true).astype(int)
    return x1, x2, p_true, y

x1_val, x2_val, p_val_true, y_val = make_dataset(n=2000, seed=1)
x1_te,  x2_te,  p_te_true,  y_te  = make_dataset(n=4000, seed=2)

## Two miscalibrated predictors

Each predictor starts from a noisy estimate of the true logit and multiplies it by a sharpness factor that depends on `x1`. A is sharpened on high-`x1` samples, B on low-`x1` samples. Both are too sharp on average, but the over-sharpening hits opposite regions of input space.

In [ ]:
def predictor_A(x1, x2, rng):
    # noisy estimate of the true logit, then sharpened on high-x1 samples
    base = true_logit(x1, x2) + rng.normal(0, 0.3, size=len(x1))
    factor = 2.0 + 1.2 * np.tanh(1.5 * x1)
    return sigmoid(factor * base)

def predictor_B(x1, x2, rng):
    # same idea, but sharpened on low-x1 samples
    base = true_logit(x1, x2) + rng.normal(0, 0.3, size=len(x1))
    factor = 2.0 - 1.2 * np.tanh(1.5 * x1)
    return sigmoid(factor * base)

pA_val = predictor_A(x1_val, x2_val, np.random.default_rng(10))
pB_val = predictor_B(x1_val, x2_val, np.random.default_rng(11))
pA_te  = predictor_A(x1_te,  x2_te,  np.random.default_rng(20))
pB_te  = predictor_B(x1_te,  x2_te,  np.random.default_rng(21))

## Save

In [ ]:
np.savez(
    "calibration_data.npz",
    y_val=y_val, y_te=y_te,
    pA_val=pA_val, pB_val=pB_val,
    pA_te=pA_te,   pB_te=pB_te,
)
print(f"validation: {len(y_val)} samples, positive rate = {y_val.mean():.3f}")
print(f"test:       {len(y_te)} samples, positive rate = {y_te.mean():.3f}")